<a href="https://colab.research.google.com/github/Sakith-N/Statistical-Learning-e22252/blob/assignment-5/Foundations_of_Statistical_inference_%26_Hypothesis_Testing_(B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Rigorous Statement of Slutsky's Theorem
Let $X_n$ and $Y_n$ be sequences of random vectors/matrices. If $X_n$ converges in distribution to a random variable $X$ ($X_n \xrightarrow{d} X$), and $Y_n$ converges in probability to a constant matrix $c$ ($Y_n \xrightarrow{p} c$), then:$$X_n + Y_n \xrightarrow{d} X + c$$ $$Y_n X_n \xrightarrow{d} c X$$
Furthermore, if $c$ is invertible, $Y_n^{-1} X_n \xrightarrow{d} c^{-1} X$.

2. he Multivariate Central Limit Theorem (CLT) establishes that:$$\sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \mathcal{N}(0, \Sigma)$$
$$ \Sigma^{-1/2} \sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \mathcal{N}(0, I_m)$$By the Weak Law of Large Numbers (WLLN), the empirical sample covariance matrix $S$ is a consistent estimator of the latent matrix $\Sigma$, meaning $S \xrightarrow{p} \Sigma$ as $n \to \infty$.
$S^{-1/2} \xrightarrow{p} \Sigma^{-1/2} \implies S^{1/2}\Sigma^{-1/2} \xrightarrow{p} I_m$
We now apply Slutsky's Theorem by combining our consistent tracking matrix with our CLT distribution sequence:$$\sqrt{n}S^{-1/2}(\hat{\mu}_n - \mu) = \left(S^{-1/2}\Sigma^{1/2}\right) \cdot \left(\Sigma^{-1/2}\sqrt{n}(\hat{\mu}_n - \mu)\right)$$$$\text{Since } S^{-1/2}\Sigma^{1/2} \xrightarrow{p} I_m \text{ and } \Sigma^{-1/2}\sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \mathcal{N}(0, I_m),$$$$\sqrt{n}S^{-1/2}(\hat{\mu}_n - \mu) \xrightarrow{d} I_m \cdot \mathcal{N}(0, I_m) \sim \mathcal{N}(0, I_m)$$As $n \to \infty$, we can rearrange this asymptotic pivoting distribution to establish the operational parametric representation:$$\hat{\mu}_n \sim \mathcal{N}\left(\mu, \frac{1}{n}S\right)$$

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010
df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    n = len(df)
    m = df.shape[1]
    chunk_size = n // g_chunks

    chunks = [df.iloc[j * chunk_size : (j + 1) * chunk_size].values for j in range(g_chunks)]

    global_mean = df.mean().values
    chunk_means = [np.mean(chunk, axis=0) for chunk in chunks]

    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for j in range(g_chunks):
        chunk = chunks[j]
        mean_j = chunk_means[j]

        diff_W = chunk - mean_j
        W += diff_W.T @ diff_W

        diff_B = mean_j - global_mean
        B += chunk_size * np.outer(diff_B, diff_B)

    det_W = np.linalg.det(W)
    det_Total = np.linalg.det(B + W)
    wilks_lambda = det_W / det_Total

    multiplier = -(n - 1 - (m + g_chunks) / 2.0)
    chi2_calc = multiplier * np.log(wilks_lambda)

    df_chi2 = m * (g_chunks - 1)
    p_value = stats.chi2.sf(chi2_calc, df_chi2)

    conclusion = (
        "Reject H0: First moment is NOT homogeneous. "
        "The baseline center has shifted over time due to operational structural drift."
        if p_value < 0.05 else
        "Fail to reject H0: First moment is homogeneous across the timeline."
    )

    return {
        "Wilks' Lambda": float(wilks_lambda),
        "Bartlett Chi-Square Stat": float(chi2_calc),
        "p-value": float(p_value),
        "Diagnostic Conclusion": conclusion
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
for key, value in results.items():
    print(f"{key}: {value}")

Wilks' Lambda: 0.9904519796929591
Bartlett Chi-Square Stat: 47.9215049961488
p-value: 3.225525950915837e-06
Diagnostic Conclusion: Reject H0: First moment is NOT homogeneous. The baseline center has shifted over time due to operational structural drift.
